# A Three-Layer Framework for Fake News Detection, Verification, and Explanation

**ECS7036P Group 10 — Notebook Scaffold**

This notebook is a guide, not an implementation. Every section below describes what
needs to happen and which GitHub issue it corresponds to, but contains no code —
fill in the empty cells as you implement each piece. Delete this line and the guide
text once a section is implemented, or move the guide text into a comment above your code.

Run top to bottom: **Setup → Data → Layer 1 (Classifier) → Layer 2 (Verification)
→ Layer 3 (Explanation) → End-to-end demo**.

## 0. Setup
*Owner: Kayleb — [Issue #1](https://github.com/KP-365/Fake_news/issues/1)*

- Confirm you're on a Colab **T4 GPU** runtime (`Runtime → Change runtime type`).
- Install dependencies from `requirements.txt`: `transformers`, `peft`, `torch`,
  `datasets`, `scikit-learn`, `gradio`, `anthropic`, `numpy`, `matplotlib`.
- Set any API keys you'll need later (Google Fact Check Tools API key, Hugging Face
  token) as Colab secrets rather than hardcoding them.

## 1. Data
*Owner: Eric — [Issue #2](https://github.com/KP-365/Fake_news/issues/2), [Issue #9](https://github.com/KP-365/Fake_news/issues/9)*

- Load the **LIAR** dataset (primary, 12,836 labelled political statements from
  PolitiFact) and the **WELFake** dataset (backup, 72,134 labelled articles).
- Build the preprocessing/tokenisation pipeline shared by both datasets so they can
  feed the same classifier interface.
- Produce a train/validation/held-out test split — the test set is what every layer's
  evaluation below will be measured against.

## 2. Layer 1 — Classification (BERT-LoRA)
*Owner: Kayleb — [Issue #3](https://github.com/KP-365/Fake_news/issues/3)*

- Load `bert-base-uncased` from Hugging Face Transformers.
- Attach a LoRA adapter via Hugging Face **PEFT** and fine-tune on the LIAR training
  split to classify a claim as real or fake.
- Keep the base weights frozen — only the LoRA low-rank updates should be trainable.

## 3. Layer 1 — Monte Carlo Dropout (uncertainty)
*Owner: Kayleb — [Issue #4](https://github.com/KP-365/Fake_news/issues/4)*

- Keep dropout active at inference time and run multiple stochastic forward passes
  per input.
- Use the spread across passes to produce a per-prediction uncertainty estimate,
  so the model can say "uncertain" instead of forcing a binary label.

## 4. Layer 1 — Evaluation: F1, confusion matrix, calibration
*Owner: Kayleb — [Issue #5](https://github.com/KP-365/Fake_news/issues/5), [Issue #6](https://github.com/KP-365/Fake_news/issues/6), [Issue #7](https://github.com/KP-365/Fake_news/issues/7)*

- Compute per-class precision/recall, macro-F1, and a 2x2 confusion matrix on the
  held-out LIAR test set. Target: macro-F1 competitive with published LIAR baselines.
- Plot a reliability diagram and compute Expected Calibration Error (ECE); compare
  against an uncalibrated softmax baseline. Target: lower ECE.
- Evaluate accuracy-on-retained predictions as the least-confident cases are deferred
  (selective classification) — confirm accuracy rises as coverage shrinks.

## 5. Layer 2 — Fact Check verification
*Owner: Eric — [Issue #10](https://github.com/KP-365/Fake_news/issues/10)*

- Query the **Google Fact Check Tools API** for each test claim, searching verified
  sources (PolitiFact, Snopes, etc.) for supporting or refuting evidence.
- Store the retrieved verdict alongside the classifier's prediction for the next step.

## 6. Layer 2 — Conflict detection & evaluation
*Owner: Eric — [Issue #11](https://github.com/KP-365/Fake_news/issues/11), [Issue #12](https://github.com/KP-365/Fake_news/issues/12)*

- Flag cases where the classifier's prediction disagrees with the retrieved
  fact-check verdict.
- Report the verification layer's coverage of the test claims and analyse the
  conflict cases as the hardest examples.

## 7. Baseline models
*Owner: William — [Issue #14](https://github.com/KP-365/Fake_news/issues/14)*

- Implement one or more baseline classifiers (e.g. TF-IDF + logistic regression, or a
  non-LoRA fine-tuned model) purely for comparison against BERT-LoRA.
- Produce evaluation plots comparing the baseline to Layer 1.

## 8. Layer 3 — Explanation agent
*Owner: William — [Issue #15](https://github.com/KP-365/Fake_news/issues/15)*

- Combine the Layer 1 signal (prediction + uncertainty) and the Layer 2 signal
  (fact-check evidence / conflict flag) into a single natural-language explanation.
- The explanation should say *why* the system reached its verdict, not just restate
  the label.

## 9. Layer 3 — Gradio UI + Hugging Face Spaces
*Owner: William — [Issue #16](https://github.com/KP-365/Fake_news/issues/16)*

- Build a Gradio interface: input a claim, output the prediction, calibrated
  confidence, any conflicting fact-checks, and the generated explanation.
- Deploy to Hugging Face Spaces to get the public demo URL required for the
  project's success criteria.

## 10. End-to-end check & faithfulness review
*Owner: Team — [Issue #19](https://github.com/KP-365/Fake_news/issues/19), [Issue #20](https://github.com/KP-365/Fake_news/issues/20)*

- Run a handful of claims through the full pipeline (classify → confidence →
  conflicts → explanation) and sanity-check the output.
- Spot-check a sample of explanations against the underlying signals to confirm
  each justification is faithful, not just plausible-sounding.

---
Full task tracker: [TASKS.md](../TASKS.md) · [GitHub Issues](https://github.com/KP-365/Fake_news/issues) · [Project board](https://github.com/users/KP-365/projects/8)